In [1]:
!pip install openpyxl -q

import pandas as pd
import numpy as np
import zipfile
import glob
import os
import re
import requests
from google.colab import drive
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, f1_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

In [12]:
extract_base = 'content/ae_data'
os.makedirs(extract_base, exist_ok=True)

!wget -O data1.zip "https://github.com/lcl-811/seem3650-project-data-files/releases/download/v.1.0/1_2026.zip"
!wget -O data2.zip "https://github.com/lcl-811/seem3650-project-data-files/releases/download/v.1.0/2_2026.zip"
!wget -O data3.zip "https://github.com/lcl-811/seem3650-project-data-files/releases/download/v.1.0/3_2026.zip"

zip_files = ['data1.zip', 'data2.zip', 'data3.zip']

--2026-05-03 14:55:48--  https://github.com/lcl-811/seem3650-project-data-files/releases/download/v.1.0/1_2026.zip
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/1228035184/268b4f79-9fc2-4a07-b350-73375e47c32c?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-05-03T15%3A44%3A25Z&rscd=attachment%3B+filename%3D1_2026.zip&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-05-03T14%3A43%3A27Z&ske=2026-05-03T15%3A44%3A25Z&sks=b&skv=2018-11-09&sig=H3KdbYRk%2BgN%2BDwsJMTaK0BlyhrZE%2BH6eghQ8U7aEBoM%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3NzgyMTk0OCwibmJmIjoxNzc3ODIwMTQ4LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZH

In [13]:
for zip_path in zip_files:
    with zipfile.ZipFile(zip_path,'r') as zf:
        zf.extractall(extract_base)

excel_files = sorted(glob.glob(f'{extract_base}/**/*.xlsx', recursive=True))
print(f"Total Excel files found: {len(excel_files)}")

Total Excel files found: 8588


In [14]:
def read_from_filename(file_path):
    base = os.path.basename(file_path)
    yyyymmdd = base.split('-')[0]
    hhmm = base.split('-')[1]
    timestamp = pd.to_datetime(f"{yyyymmdd} {hhmm[:2]}:{hhmm[2:]}", format='%Y%m%d %H:%M')
    df = pd.read_excel(file_path, header=0, skiprows=8, usecols='A:G')
    df.columns = ['hospital', 'wait_I', 'treating_I', 'wait_II', 'treating_II', 'wait_III', 'wait_IV_V']
    df['timestamp'] = timestamp
    return df

In [15]:
all_data = []
total = len(excel_files)
for i, fp in enumerate(excel_files):
    try:
        df = read_from_filename(fp)
        # Quick check: ensure timestamp is not None
        if df['timestamp'].iloc[0] is None:
            print(f"Warning: timestamp None for {fp}")
        all_data.append(df)
    except Exception as e:
        print(f"Error on {fp}: {e}")
    if (i+1) % 1000 == 0:
        print(f"Processed {i+1}/{total} files")

combined = pd.concat(all_data, ignore_index=True)
print(f"Total rows: {len(combined)}")
print(f"Date range: {combined['timestamp'].min()} to {combined['timestamp'].max()}")

Processed 1000/8588 files
Processed 2000/8588 files
Processed 3000/8588 files
Processed 4000/8588 files
Processed 5000/8588 files
Processed 6000/8588 files
Processed 7000/8588 files
Processed 8000/8588 files
Total rows: 197494
Date range: 2026-01-01 00:00:00 to 2026-03-31 23:45:00


After extract and combine all the files from 1/1/2026 to 31/3/2026, we need to convert all the waiting time to minutes first.

In [16]:
def parse_median(text):
    if pd.isna(text):
        return np.nan
    text = str(text)
    nums = re.findall(r'\d+', text)
    if not nums:
        return np.nan
    if 'hour' in text.lower():
        return int(nums[0]) * 60
    else:
        return int(nums[0])

combined['wait_IV_V_med'] = combined['wait_IV_V'].apply(parse_median)
combined['treating_I'] = combined['treating_I'].map({'Y':1,'N':0}).fillna(0)
combined['treating_II'] = combined['treating_II'].map({'Y':1,'N':0}).fillna(0)
combined['resus_overload'] = combined['treating_I'].isna().astype(int) | combined['treating_II'].isna().astype(int)
combined['treating_I'] = combined['treating_I'].fillna(0)
combined['treating_II'] = combined['treating_II'].fillna(0)
combined['wait_IV_V_med'] = combined['wait_IV_V_med'].clip(upper=720)

Then sorts the combined DataFrame by hospital and timestamp to ensure correct order for lagging.
For each hospital, we creates
lag1: median waiting time from 15 minutes earlier(shift 1 step).
lag4: median waiting time from 1 hour earlier(shift 4 steps, because 4×15 min=1 hour).
roll_mean4: rolling average of median waiting time over the past 4 observations.

In [17]:
combined = combined.sort_values(['hospital','timestamp'])
df = combined.copy()
hospitals = df['hospital'].unique()
for h in hospitals:
    mask = df['hospital'] == h
    df.loc[mask, 'lag1'] = df.loc[mask, 'wait_IV_V_med'].shift(1)
    df.loc[mask, 'lag4'] = df.loc[mask, 'wait_IV_V_med'].shift(4)
    df.loc[mask, 'roll_mean4'] = df.loc[mask, 'wait_IV_V_med'].rolling(4, min_periods=1).mean()
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

crowded: 1 if median waiting time > 120 minutes.
target: shifted crowded by -2 rows → predicts crowding 30 minutes ahead (since each row = 15 min).
dropna: removes rows without a future label.
mean() prints the proportion of positive crowding cases in the final dataset.



In [18]:
df['crowded'] = (df['wait_IV_V_med'] > 120).astype(int)
df['target'] = df.groupby('hospital')['crowded'].shift(-2)
model_df = df.dropna(subset=['target']).copy()
print(f"Crowded rate: {model_df['target'].mean():.3f}")

Crowded rate: 0.281



*   Splits data by date: **train = before March 2026, test = March 2026**.

*   Selects 9 features: two lags (15 min & 1 hour), a rolling average, treatment indicators, resuscitation flag, and time features (hour, day of week, weekend).

*   Fills missing values with 0, separates features (X) and target (y) for both sets.










In [19]:
train=model_df[model_df['timestamp']<'2026-03-01']
test=model_df[model_df['timestamp']>='2026-03-01']

feat_cols=['lag1', 'lag4', 'roll_mean4', 'treating_I', 'treating_II', 'resus_overload', 'hour', 'day_of_week', 'is_weekend']
X_train=train[feat_cols].fillna(0)
y_train=train['target']
X_test=test[feat_cols].fillna(0)
y_test=test['target']

* Compute 'scale_pos_weight' = (# negative / # positive) on the training set to handle class imbalance

* Train XGBoost classifier with this 'scale_pos_weight' on '(X_train, y_train)

* Train two baseline models on the same features: logistic regression and random forest, both with 'class_weight='balanced''

In [20]:
scale=(y_train==0).sum()/(y_train==1).sum() if(y_train==1).sum()>0 else 1

xgb_model = XGBClassifier(
    scale_pos_weight=scale,
    eval_metric='logloss',
    random_state=42)
xgb_model.fit(X_train, y_train)

#logistic regression baseline

lr_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
lr_model.fit(X_train, y_train)

#random forest baseline
rf_model = RandomForestClassifier(
    class_weight='balanced',
    n_estimators=200,
    random_state=42
)
rf_model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=200,
                       random_state=42)

* We evaluate the 3 models on March 2026 test set using AUC-ROC, PR-AUC, recall for the crowded class and F1-score.

* For each model, we take the predicted probabilities on the test set and apply a 0.5 threshold to obtain class labels.

* At last, report the metrics in a comparision table.

In [21]:
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, f1_score
import pandas as pd

y_prob_lr = lr_model.predict_proba(X_test)[:, 1]
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

def get_metrics(y_true, y_prob, thresold=0.5):
  y_pred = (y_prob >= thresold).astype(int)
  return {
      "AUC_ROC": roc_auc_score(y_true, y_prob),
      "PR_AUC": average_precision_score(y_true, y_prob),
      "Recall_crowded": recall_score(y_true, y_pred),
      "F1": f1_score(y_true, y_pred)
  }

metrics_lr = get_metrics(y_test, y_prob_lr)
metrics_rf = get_metrics(y_test, y_prob_rf)
metrics_xgb = get_metrics(y_test, y_prob_xgb)

results = pd.DataFrame({
    "LogisticRegression": metrics_lr,
    "RandomForest": metrics_rf,
    "XGBoost": metrics_xgb,
}).T.round(4)

print(results)

                    AUC_ROC  PR_AUC  Recall_crowded      F1
LogisticRegression   0.9645  0.9317          0.9085  0.8398
RandomForest         0.9515  0.9044          0.8641  0.8265
XGBoost              0.9685  0.9375          0.9218  0.8485
